In [13]:
import numpy as np
import pandas as pd

In [14]:
df = pd.read_csv(
    r"F:\CodexProjects\光斑图片\DataBase\analysis\beam_m2_per_position.csv"
)

lam_um = 10.6


def compute_m2(z_mm, d4sigma_um):
    z_um = z_mm * 1000.0
    # d4sigma 为 4-sigma 光斑直径，转换为半径应除以 2
    rho_um = d4sigma_um / 2.0
    coeff = np.polyfit(z_um, rho_um**2, 2)
    a, b, c = coeff
    z0_um = -b / (2 * a)
    rho0_sq = c - b * b / (4 * a)
    rho0_um = np.sqrt(rho0_sq)
    theta_rad = np.sqrt(a)
    m2 = np.pi * rho0_um * theta_rad / lam_um
    return {
        "a": a,
        "b": b,
        "c": c,
        "z0_um": z0_um,
        "rho0_um": rho0_um,
        "theta_rad": theta_rad,
        "M2": m2,
    }


cols = [c for c in df.columns if "d4" in c and c.endswith("_um")]
for col in cols:
    result = compute_m2(df["z_mm"].values, df[col].values)
    print(
        f'{col}: M2={result["M2"]:.4f}, z0={result["z0_um"]:.1f} um, rho0={result["rho0_um"]:.1f} um'
    )

fit_df = pd.read_csv(
    r"F:\CodexProjects\光斑图片\DataBase\analysis\beam_m2_fit_summary.csv"
)
fit_df.columns = fit_df.columns.str.strip()
print("\nbeam_m2_fit_summary.csv:")
print(
    fit_df[
        ["dataset", "direction", "m2", "d0_um", "theta_um_per_mm", "z0_mm"]
    ].to_string(index=False)
)
print()


def compute_m2_from_summary(row):
    rho0_um = row.d0_um / 2.0
    theta_rad = row.theta_um_per_mm / 2000.0
    m2_calc = np.pi * rho0_um * theta_rad / lam_um
    return m2_calc, rho0_um, theta_rad


fit_df["m2_calc"], fit_df["rho0_calc_um"], fit_df["theta_calc_rad"] = zip(
    *fit_df.apply(compute_m2_from_summary, axis=1)
)

print("summary M2 comparison:")
print(
    fit_df[
        ["dataset", "direction", "m2", "m2_calc", "d0_um", "theta_um_per_mm", "z0_mm"]
    ].to_string(index=False)
)
print()
for row in fit_df.itertuples(index=False):
    print(
        f"{row.dataset} {row.direction}: M2_summary={row.m2:.6f}, M2_calc={row.m2_calc:.6f}, "
        f"delta={row.m2_calc-row.m2:.6f}, d0={row.d0_um:.1f} um, "
        f"theta_rad={row.theta_calc_rad:.6f} rad, z0={row.z0_mm:.3f} mm"
    )

csv_d4x_um: M2=2.6339, z0=234273.9 um, rho0=451.4 um
csv_d4y_um: M2=1.6090, z0=234643.0 um, rho0=250.2 um
png_original_moment_d4x_um: M2=1.3815, z0=222188.3 um, rho0=290.9 um
png_original_moment_d4y_um: M2=0.6249, z0=213512.2 um, rho0=150.7 um
png90_d4x_um: M2=0.8555, z0=220656.1 um, rho0=194.5 um
png90_d4y_um: M2=0.6065, z0=215703.6 um, rho0=160.2 um
inferred90_d4x_um: M2=2.0145, z0=232357.7 um, rho0=375.2 um
inferred90_d4y_um: M2=0.7735, z0=237820.3 um, rho0=128.5 um

beam_m2_fit_summary.csv:
                  dataset  direction       m2      d0_um  theta_um_per_mm      z0_mm
csv_original               x         2.606499 891.189644        39.473304 234.933974
csv_original               y         1.655201 515.010646        43.376103 235.321664
png90_shape_inferred       x         2.068380 771.363834        36.189875 233.801593
png90_shape_inferred       y         1.197097 401.498143        40.240367 236.241321
png90                      x         1.409687 654.328854        29.076539 2

In [15]:
fit_df = pd.read_csv(
    r"F:\CodexProjects\光斑图片\DataBase\analysis\beam_m2_fit_summary.csv"
)
fit_df.columns = fit_df.columns.str.strip()
print("\nbeam_m2_fit_summary.csv:")
print(
    fit_df[
        ["dataset", "direction", "m2", "d0_um", "theta_um_per_mm", "z0_mm"]
    ].to_string(index=False)
)
print()
for row in fit_df.itertuples(index=False):
    print(
        f"{row.dataset} {row.direction}: M2={row.m2:.6f}, d0={row.d0_um:.1f} um, "
        f"theta={row.theta_um_per_mm:.4f} um/mm, z0={row.z0_mm:.3f} mm"
    )


beam_m2_fit_summary.csv:
                  dataset  direction       m2      d0_um  theta_um_per_mm      z0_mm
csv_original               x         2.606499 891.189644        39.473304 234.933974
csv_original               y         1.655201 515.010646        43.376103 235.321664
png90_shape_inferred       x         2.068380 771.363834        36.189875 233.801593
png90_shape_inferred       y         1.197097 401.498143        40.240367 236.241321
png90                      x         1.409687 654.328854        29.076539 226.497139
png90                      y         1.054248 564.814992        25.191422 224.665780
png_original_moment_check  x         1.777457 758.565123        31.624387 227.437880
png_original_moment_check  y         1.358935 673.803671        27.219573 223.258333

csv_original               x        : M2=2.606499, d0=891.2 um, theta=39.4733 um/mm, z0=234.934 mm
csv_original               y        : M2=1.655201, d0=515.0 um, theta=43.3761 um/mm, z0=235.322 mm
png90_shap

In [16]:
fit_df = pd.read_csv(
    r"F:\CodexProjects\光斑图片\DataBase\analysis\beam_m2_fit_summary.csv"
)
fit_df.columns = fit_df.columns.str.strip()
print("\nbeam_m2_fit_summary.csv:")
print(
    fit_df[
        ["dataset", "direction", "m2", "d0_um", "theta_um_per_mm", "z0_mm"]
    ].to_string(index=False)
)
print()
for row in fit_df.itertuples(index=False):
    print(
        f"{row.dataset} {row.direction}: M2={row.m2:.6f}, d0={row.d0_um:.1f} um, "
        f"theta={row.theta_um_per_mm:.4f} um/mm, z0={row.z0_mm:.3f} mm"
    )


beam_m2_fit_summary.csv:
                  dataset  direction       m2      d0_um  theta_um_per_mm      z0_mm
csv_original               x         2.606499 891.189644        39.473304 234.933974
csv_original               y         1.655201 515.010646        43.376103 235.321664
png90_shape_inferred       x         2.068380 771.363834        36.189875 233.801593
png90_shape_inferred       y         1.197097 401.498143        40.240367 236.241321
png90                      x         1.409687 654.328854        29.076539 226.497139
png90                      y         1.054248 564.814992        25.191422 224.665780
png_original_moment_check  x         1.777457 758.565123        31.624387 227.437880
png_original_moment_check  y         1.358935 673.803671        27.219573 223.258333

csv_original               x        : M2=2.606499, d0=891.2 um, theta=39.4733 um/mm, z0=234.934 mm
csv_original               y        : M2=1.655201, d0=515.0 um, theta=43.3761 um/mm, z0=235.322 mm
png90_shap